# 06 · Panel municipal: deforestación, crédito y producción

**Qué hace.** Junta, por municipio, la deforestación de las tablas propias (IDEAM y alertas de GFW) con las
tablas del Observatorio: el crédito de FINAGRO, los cultivos de la EVA y el ganado del ICA. Es la tabla que
usan el análisis exploratorio, la segmentación y los modelos.

**Cuándo correrlo.** Después de los notebooks 03 y 04, y cada vez que se actualice cualquiera de ellos o
alguna tabla del Observatorio.

**Qué lee.** De `DATASET_DEFORESTACION`, las tablas `PANEL_IDEAM` y `PANEL_DEFORESTACION_COLOMBIA`. Del
Observatorio, las tablas de FINAGRO, EVA, ICA y municipios de la celda 1.

**Qué escribe**, en `DATASET_DEFORESTACION`:

| Tabla | Qué es |
|---|---|
| `MUNICIPIO_ANIO` | Una fila por municipio y año: deforestación, crédito, cultivos, ganado y atributos del municipio |
| `MUNICIPIO_MES` | Una fila por municipio y mes: alertas de GFW y crédito |
| `MUNICIPIO_CULTIVO_ANIO` | Una fila por municipio, cultivo y año: hectáreas cosechadas según la EVA |

**Qué tienes que cambiar.** Nada: la celda 1 ya apunta al proyecto del Observatorio (`prueba-ofr`) y a
sus tablas.

**Cuánto tarda.** Unos cinco minutos.

In [ ]:
# =====================================================================
# 1. CONFIGURACIÓN
#    Ya está llena para el proyecto del Observatorio (prueba-ofr). Solo
#    hay que cambiarla si las tablas se pasan a otro proyecto o dataset.
# =====================================================================
# --- Dónde están las tablas en BigQuery ------------------------------
# Proyecto de Google Cloud del Observatorio (Prueba-OFR). Es el id,
# no el nombre: el que aparece en la consola al elegir el proyecto.
PROYECTO = "prueba-ofr"

# Dataset con las tablas de deforestación que arman los notebooks 01
# a 06 (grilla, paneles y panel municipal).
DATASET_DEFORESTACION = "Cambio_Climatico"

# Ubicación de los datos. Debe ser la misma de las tablas del
# Observatorio (en BigQuery: abre una tabla > Detalles > Ubicación de
# los datos). Las del Observatorio están hoy en US.
UBICACION = "US"

# --- Tablas del Observatorio (este notebook solo las lee) -------------
# Nombre completo: proyecto.dataset.tabla. Son las que se usaron en el
# trabajo de grado.
# Tabla de desembolsos de FINAGRO
TABLA_FINAGRO = "prueba-ofr.Inclusion_Financiera.FINAGRO_Desembolsos_EFECTIVA"
# Evaluaciones Agropecuarias Municipales (EVA)
TABLA_EVA = "prueba-ofr.Produccion.eva_full_enriquecido"
# Inventario pecuario del ICA
TABLA_ICA = "prueba-ofr.Produccion.ICA_pecuario"
# Catálogo de municipios
TABLA_MUNICIPIOS = "prueba-ofr.Municipios.CODIGO"

# --- Para actualizar --------------------------------------------------
# Años de municipio_anio. El último debe ser un año completo en todas las
# fuentes de deforestación; en el trabajo, 2026 iba incompleto.
ANIO_INICIO = 2020
ANIO_FIN = 2025

# Último año que trae la EVA. Hasta ese año, un municipio sin cultivos
# reportados cuenta como cero; después, queda vacío (no hay dato).
ULTIMO_ANIO_EVA = 2024

# Meses de municipio_mes: hasta el último mes completo en GFW y FINAGRO.
MES_INICIO = "2020-01-01"
MES_FIN = "2026-06-01"          # incluido

# Primer año de municipio_cultivo_anio. Desde 2019 la EVA usa los mismos
# nombres de grupo y de cultivo todos los años; antes usaba otros.
ANIO_INICIO_CULTIVOS = 2019

# --- Solo si corres el notebook en tu computador ----------------------
# En BigQuery déjalo vacío. En un computador: la ruta al archivo .json
# de la llave de una cuenta de servicio con acceso a estas tablas.
RUTA_LLAVE_BIGQUERY = ""

In [ ]:
# =====================================================================
# 2. INSTALAR PAQUETES
#    En BigQuery instala lo que le falte a la máquina; la primera vez
#    tarda uno o dos minutos. No hay que cambiar nada. En un computador
#    con el entorno del repositorio (requirements_local.txt) se puede
#    saltar.
#    Las versiones fijas son las que se usaron en el trabajo de grado,
#    para que los resultados salgan iguales.
# =====================================================================
%pip install --quiet pyarrow pandas-gbq

In [ ]:
# =====================================================================
# 3. CONEXIÓN CON BIGQUERY Y FUNCIONES DE APOYO
#    No hay que cambiar nada en esta celda. Se conecta a BigQuery y
#    define las funciones con que el resto del notebook lee y escribe
#    tablas. Es igual en todos los notebooks de esta carpeta.
# =====================================================================
import json
import re
import tempfile
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from google.cloud import bigquery

NOMBRE_NOTEBOOK = "06_carga_panel_municipal"

# --- 3.1 ¿Quedó algo sin llenar en la celda 1? --------------------------
# Si queda algún <<LLENAR ...>>, el notebook se detiene aquí y dice cuál.
_pendientes = sorted(n for n, v in list(globals().items())
                     if n.isupper() and isinstance(v, str) and "<<LLENAR" in v)
if _pendientes:
    raise ValueError("Falta llenar en la celda 1: " + ", ".join(_pendientes))

# --- 3.2 Conexión ---------------------------------------------------------
# En BigQuery el notebook entra con la cuenta de quien lo corre y no
# necesita llave. La llave solo se usa si se corre en un computador.
if globals().get("RUTA_LLAVE_BIGQUERY"):
    from google.oauth2 import service_account
    client = bigquery.Client(
        project=PROYECTO,
        credentials=service_account.Credentials.from_service_account_file(RUTA_LLAVE_BIGQUERY))
else:
    client = bigquery.Client(project=PROYECTO)
print(f"Conectado a BigQuery: proyecto {PROYECTO}")

# Carpeta para los archivos que se descargan mientras corre el notebook.
# Es de paso: todo lo que importa termina en tablas de BigQuery.
_carpeta = globals().get("CARPETA_DESCARGAS") or ""
DIR_TRABAJO = Path(_carpeta) if _carpeta else Path(tempfile.gettempdir()) / "ofr_deforestacion"
DIR_TRABAJO.mkdir(parents=True, exist_ok=True)


# --- 3.3 Nombres de tabla -----------------------------------------------
def nombre_completo(tabla, dataset=None):
    """'panel_ideam' -> 'proyecto.dataset.PANEL_IDEAM'.

    Las tablas que crean estos notebooks se nombran en mayúscula
    (GRILLA, PANEL_IDEAM...). Si la tabla ya viene completa
    (proyecto.dataset.tabla, como las del Observatorio), se devuelve igual.
    Si no, se arma con PROYECTO y el dataset indicado; por defecto,
    DATASET_DEFORESTACION.
    """
    if tabla.count(".") == 2:
        return tabla
    return f"{PROYECTO}.{dataset or globals().get('DATASET_DEFORESTACION')}.{tabla.upper()}"


# --- 3.4 Leer -------------------------------------------------------------
_MARCA = re.compile(r"^(\[índice(?: sin nombre)?\] )?(?:Nombre original: «(.*?)»)?")


_LECTURA_RAPIDA = [True]


def leer_sql(sql):
    """Corre una consulta SQL en BigQuery y devuelve un DataFrame de pandas.

    Usa la lectura rápida de BigQuery (Storage Read API). Si no está
    disponible (por ejemplo, porque la cuenta no tiene el rol «BigQuery
    Read Session User»), sigue con la lectura normal, que da el mismo
    resultado pero tarda más en tablas grandes.
    """
    trabajo = client.query(sql)
    trabajo.result()                      # si la consulta tiene un error, sale aquí
    resultado = None
    if _LECTURA_RAPIDA[0]:
        try:
            resultado = trabajo.to_arrow()
        except Exception as e:
            _LECTURA_RAPIDA[0] = False
            print("Aviso: la lectura rápida de BigQuery no está disponible "
                  f"({type(e).__name__}); se usa la lectura normal.")
    if resultado is None:
        resultado = trabajo.to_arrow(create_bqstorage_client=False)
    return resultado.to_pandas(date_as_object=False, coerce_temporal_nanoseconds=True)


def leer_tabla(tabla, columnas=None, dataset=None, filtro=None):
    """Lee una tabla de BigQuery a pandas.

    columnas  lista de columnas a traer; por defecto, todas.
    filtro    condición SQL opcional, p. ej. "anio >= 2021".

    Si la tabla se guardó con guardar_tabla(), devuelve las filas en el
    mismo orden en que se guardaron, las columnas con su nombre original
    (tildes, espacios) y el índice.
    """
    destino = nombre_completo(tabla, dataset)
    esquema = client.get_table(destino).schema
    con_orden = any(c.name == "orden_fila" for c in esquema)
    seleccion = ", ".join(f"`{c}`" for c in columnas) if columnas else "*"
    df = leer_sql(f"SELECT {seleccion} FROM `{destino}`"
                  + (f" WHERE {filtro}" if filtro else "")
                  + (" ORDER BY orden_fila" if con_orden else ""))
    df = df.drop(columns=["orden_fila"], errors="ignore")

    originales, indice, sin_nombre = {}, [], set()
    for campo in esquema:
        marca = _MARCA.match(campo.description or "")
        if marca.group(2):
            originales[campo.name] = marca.group(2)
        if marca.group(1) and campo.name in df.columns:
            indice.append(campo.name)
            if "sin nombre" in marca.group(1):
                sin_nombre.add(campo.name)
    if indice:
        df = df.set_index(indice)
        df.index = df.index.set_names(
            [None if n in sin_nombre else originales.get(n, n) for n in indice])
    return df.rename(columns=originales)


# --- 3.5 Escribir -----------------------------------------------------------
_DATASETS_LISTOS = set()


def _nombre_para_bigquery(nombre, usados):
    """Nombre de columna que BigQuery acepta siempre: letras sin tilde,
    números y guion bajo. Los que ya cumplen se dejan igual."""
    texto = str(nombre)
    if re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]{0,299}", texto) and texto.lower() not in usados:
        usados.add(texto.lower())
        return texto
    limpio = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    limpio = re.sub(r"[^0-9A-Za-z]+", "_", limpio).strip("_").lower() or "columna"
    if limpio[0].isdigit():
        limpio = "c_" + limpio
    limpio, base, k = limpio[:280], limpio[:280], 2
    while limpio.lower() in usados:
        limpio, k = f"{base}_{k}", k + 1
    usados.add(limpio.lower())
    return limpio


def _tipos_para_bigquery(df):
    """Deja cada columna en un tipo que BigQuery reciba sin error:
    categorías a texto, enteros con vacíos a entero, mezclas a texto."""
    for c in df.columns:
        s = df[c]
        if isinstance(s.dtype, pd.CategoricalDtype):
            s = s.astype(object).where(s.notna(), None)
        if s.dtype == object:
            valores = s.dropna()
            tipos = {type(v) for v in valores}
            if not tipos or tipos <= {str}:
                pass
            elif all(isinstance(v, (bool, np.bool_)) for v in valores):
                s = s.astype("boolean")
            elif all(isinstance(v, (int, np.integer)) and not isinstance(v, bool) for v in valores):
                s = s.astype("Int64")
            elif all(isinstance(v, (int, float, np.integer, np.floating)) for v in valores):
                s = s.astype(float)
            else:
                s = s.map(lambda v: None if v is None or v is pd.NA
                          or (isinstance(v, float) and v != v) else str(v))
        df[c] = s
    return df


def guardar_tabla(df, tabla, descripcion, dataset=None, indice=False,
                  descripciones=None, procedencia=None):
    """Escribe `df` en BigQuery. Si la tabla ya existe, la REEMPLAZA entera.

    tabla          nombre corto, p. ej. "panel_ideam".
    descripcion    texto que se ve en BigQuery, en los detalles de la tabla.
    dataset        por defecto, DATASET_DEFORESTACION. Si no existe, se crea.
    indice         True para guardar también el índice de pandas como
                   columna(s); leer_tabla() lo recupera.
    descripciones  {columna: texto}, para documentar columnas en BigQuery.
    procedencia    {"fuente": ..., ...}: si se da, deja una fila en la tabla
                   registro_cargas con qué se cargó, de dónde y cuándo.
    """
    dataset = dataset or globals().get("DATASET_DEFORESTACION")
    destino = nombre_completo(tabla, dataset)
    if dataset not in _DATASETS_LISTOS:
        ds = bigquery.Dataset(f"{PROYECTO}.{dataset}")
        ds.location = globals().get("UBICACION", "US")
        client.create_dataset(ds, exists_ok=True)
        _DATASETS_LISTOS.add(dataset)

    datos = df.copy()
    marcas = {}
    if indice:
        nombres_indice = list(datos.index.names)
        datos = datos.reset_index()
        for k, n in enumerate(nombres_indice):
            columna = datos.columns[k]
            marcas[columna] = "[índice sin nombre] " if n is None else "[índice] "
    else:
        datos = datos.reset_index(drop=True)
    if isinstance(datos.columns, pd.MultiIndex):
        datos.columns = [" | ".join(str(x) for x in c if str(x) != "") for c in datos.columns]

    descripciones = descripciones or {}
    usados, notas, columnas = set(), {}, []
    for c in datos.columns:
        nuevo = _nombre_para_bigquery(c, usados)
        texto = marcas.get(c, "")
        if nuevo != str(c):
            texto += f"Nombre original: «{c}»"
        if descripciones.get(c):
            texto += (". " if texto and not texto.endswith("] ") else "") + descripciones[c]
        notas[nuevo] = texto[:1024]
        columnas.append(nuevo)
    datos.columns = columnas
    datos = _tipos_para_bigquery(datos)
    # BigQuery no guarda el orden de las filas, y algunos cálculos (el
    # muestreo del random forest, por ejemplo) dependen de él. Esta
    # columna lo conserva; leer_tabla() la usa y la quita.
    datos["orden_fila"] = np.arange(len(datos), dtype="int64")
    notas["orden_fila"] = ("Posición de la fila en la tabla original. Sirve para leerla "
                           "en el mismo orden: ORDER BY orden_fila.")

    trabajo = client.load_table_from_dataframe(
        datos, destino,
        job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
    trabajo.result()

    t = client.get_table(destino)
    t.description = descripcion
    t.schema = [bigquery.SchemaField(f.name, f.field_type, mode=f.mode,
                                     description=notas.get(f.name) or None,
                                     fields=f.fields) for f in t.schema]
    client.update_table(t, ["description", "schema"])
    print(f"Guardada {destino}: {t.num_rows:,} filas, {len(t.schema)} columnas")

    if procedencia is not None:
        fila = pd.DataFrame([{
            "tabla": tabla.upper(),
            "filas": int(t.num_rows),
            "fuente": str(procedencia.get("fuente", "")),
            "parametros": json.dumps({k: v for k, v in procedencia.items() if k != "fuente"},
                                     ensure_ascii=False, default=str),
            "notebook": NOMBRE_NOTEBOOK,
            "cargado_en": datetime.now(timezone.utc),
        }])
        client.load_table_from_dataframe(
            fila, nombre_completo("registro_cargas", dataset),
            job_config=bigquery.LoadJobConfig(write_disposition="WRITE_APPEND")).result()

## 4. Las tablas del Observatorio

Se le pide a BigQuery solo lo que se usa, para no bajar tablas enteras:

- **FINAGRO** (millones de filas): BigQuery suma el valor de los créditos por municipio, mes, cadena y
  eslabón. Con esa suma se arma todo lo demás.
- **EVA**, **ICA** y **municipios**: solo las columnas que se usan.

In [ ]:
# --- LECTURA DE LAS TABLAS DEL OBSERVATORIO ---
finagro = leer_sql(f"""
    SELECT cod_municipio,
           DATETIME_TRUNC(fecha_credito, MONTH) AS mes,
           cadena,
           eslabon_cadena,
           SUM(valor_credito) AS valor_credito
    FROM `{TABLA_FINAGRO}`
    GROUP BY cod_municipio, mes, cadena, eslabon_cadena
""")
eva = leer_sql(f"""
    SELECT Cod_mun, CAST(Year AS DATETIME) AS Year, Grupo_cultivo, Cultivo, Cosechada
    FROM `{TABLA_EVA}`
""")
ica = leer_sql(f"""
    SELECT codigo_municipio, anio, total_bovinos, total_fincas_con_bovinos
    FROM `{TABLA_ICA}`
""")
municipios_obs = leer_sql(f"""
    SELECT cod_dane, fecha, nombre_departamento, nombre_municipio, region, ruralidad
    FROM `{TABLA_MUNICIPIOS}`
""")
for nombre, df in [("FINAGRO, sumado por municipio y mes", finagro), ("EVA", eva), ("ICA", ica),
                   ("municipios", municipios_obs)]:
    print(f"{nombre}: {len(df):,} filas")

## 5. Preparar cada fuente

- **Crédito de FINAGRO**, en millones de pesos. Se separa el eslabón de **producción** de los de
  transformación, comercialización y servicios, y dentro de producción se siguen cinco cadenas: ganadería
  (doble propósito, leche y carne), palma de aceite, cacao, café y arroz.
- **EVA**: hectáreas **cosechadas**; los dos semestres se suman en el año. De 2006 a 2018 la EVA trae los
  códigos de Antioquia sin el cero inicial, por eso se rellenan a cinco dígitos.
- **ICA**: inventario de bovinos y de fincas con bovinos.
- **Deforestación**: los paneles vienen por celda de 5 km; se suman por municipio. El periodo del IDEAM
  "2020-2021" es el año 2021.
- **Atributos del municipio** (departamento, nombre, región, ruralidad): de la tabla de municipios del
  Observatorio, el registro más reciente de cada uno.

Los nombres de cadenas y cultivos se comparan en minúsculas y sin tildes, porque las fuentes no los
escriben siempre igual ("Ganadería carne", "Ganaderia Leche").

In [ ]:
import unicodedata


def normalizar(s):
    """Minúsculas y sin tildes, para comparar textos."""
    quitar = lambda x: unicodedata.normalize("NFKD", str(x)).encode("ascii", "ignore").decode().lower().strip()
    return s.map({v: quitar(v) for v in s.dropna().unique()})


# --- FINAGRO: crédito en millones de pesos
fin = finagro.copy()
fin["cod_dane"] = fin.cod_municipio.astype(str)
fin["valor_mm"] = fin.valor_credito / 1e6
fin["anio"] = fin.mes.dt.year
fin["produccion"] = fin.eslabon_cadena == "Producción"
cadena = normalizar(fin.cadena).astype(str)
fin["grupo"] = cadena.map({"palma de aceite": "palma", "cacao": "cacao", "cafe": "cafe", "arroz": "arroz"})
fin.loc[cadena.str.startswith("ganad"), "grupo"] = "ganaderia"


def credito(df, tiempo):
    """Crédito total, de producción y de producción por cadena, por municipio y `tiempo`."""
    llave = ["cod_dane", tiempo]
    prod = df[df.produccion]
    out = pd.DataFrame({"credito_mm": df.groupby(llave).valor_mm.sum(),
                        "credito_produccion_mm": prod.groupby(llave).valor_mm.sum()})
    por_cadena = prod.groupby(llave + ["grupo"]).valor_mm.sum().unstack()
    return out.join(por_cadena.add_prefix("credito_").add_suffix("_produccion_mm"))


# --- EVA: hectáreas cosechadas
eva_anual = eva[["Cod_mun", "Year", "Cultivo", "Cosechada"]].copy()
eva_anual["cod_dane"] = eva_anual.Cod_mun.astype(str).str.zfill(5)
eva_anual["anio"] = eva_anual.Year.dt.year
eva_anual["cultivo"] = normalizar(eva_anual.Cultivo)

# --- ICA: inventario bovino
ica_anual = ica.copy()
ica_anual["cod_dane"] = ica_anual.codigo_municipio.astype(str)

# --- Deforestación: los paneles por celda se suman por municipio
ideam = leer_tabla("panel_ideam", columnas=["cod_dane", "cell_id", "periodo", "def_ha", "bosque_ideam_ha",
                                            "sin_info_ha"])
ideam["anio"] = ideam.periodo.astype(str).str[-4:].astype(int)       # "2020-2021" -> 2021
gfw = leer_sql(f"""
    SELECT cod_dane, periodo, anio, SUM(area_def_ha) AS area_def_ha
    FROM `{nombre_completo('panel_deforestacion_colombia')}`
    GROUP BY cod_dane, periodo, anio
""")
gfw["periodo"] = gfw.periodo.astype("datetime64[ns]")

# --- Atributos del municipio: el registro más reciente
municipios = (municipios_obs.assign(cod_dane=lambda d: d.cod_dane.astype(str))
              .sort_values("fecha").groupby("cod_dane")
              [["nombre_departamento", "nombre_municipio", "region", "ruralidad"]].last())

# Bosque del IDEAM al inicio del primer periodo y área de la grilla de cada municipio
primer_anio = ideam[ideam.anio == ideam.anio.min()]
fijas = primer_anio.groupby("cod_dane").agg(bosque_ideam_inicial_ha=("bosque_ideam_ha", "sum"),
                                            n_celdas=("cell_id", "size"))
fijas["area_grilla_ha"] = fijas.n_celdas * 2500            # cada celda son 5 x 5 km = 2.500 ha

# El 27493 (Belén de Bajirá, Chocó) está en los paneles y en la EVA, pero no en la tabla de municipios
# del Observatorio: se une la lista de los dos lados para no perderlo.
codigos = municipios.index.union(fijas.index)
print(f"Municipios: {len(codigos):,}")

## 6. Municipio × año

Una fila por municipio y año, de `ANIO_INICIO` a `ANIO_FIN`. Donde un municipio no tiene créditos ese año,
el crédito es cero; lo mismo con los cultivos, pero solo hasta el último año que cubre la EVA.

`tasa_def_ideam` = deforestación del IDEAM / bosque natural al inicio del periodo (`bosque_ideam_ha`): la
variable que predicen los modelos.

El chequeo final dice qué parte de cada fuente quedó en la tabla. No es 100 % porque las fuentes traen
también municipios o años que el panel no cubre.

In [ ]:
ANIOS = range(ANIO_INICIO, ANIO_FIN + 1)
llave = ["cod_dane", "anio"]

deforestacion = pd.concat([
    ideam.groupby(llave)[["def_ha", "bosque_ideam_ha", "sin_info_ha"]].sum()
         .rename(columns={"def_ha": "def_ideam_ha", "sin_info_ha": "sin_info_ideam_ha"}),
    gfw.groupby(llave).area_def_ha.sum().rename("alertas_gfw_ha"),
], axis=1)

agricola = eva_anual.groupby(llave).agg(area_cosechada_ha=("Cosechada", "sum"))
for cultivo, nombre in {"palma de aceite": "palma", "cacao": "cacao", "cafe": "cafe", "arroz": "arroz"}.items():
    agricola[f"cosechada_{nombre}_ha"] = eva_anual[eva_anual.cultivo == cultivo].groupby(llave).Cosechada.sum()

pecuario = ica_anual.groupby(llave)[["total_bovinos", "total_fincas_con_bovinos"]].sum().rename(
    columns={"total_bovinos": "bovinos", "total_fincas_con_bovinos": "fincas_bovinas"})

esqueleto = pd.MultiIndex.from_product([codigos, ANIOS], names=llave)
anual = pd.DataFrame(index=esqueleto).join(deforestacion).join(credito(fin, "anio")).join(agricola).join(pecuario)

col_credito = [c for c in anual if c.startswith(("credito_", "n_creditos"))]
col_agricola = list(agricola.columns)
anual[col_credito] = anual[col_credito].fillna(0)
cubre_eva = anual.index.get_level_values("anio") <= ULTIMO_ANIO_EVA
anual.loc[cubre_eva, col_agricola] = anual.loc[cubre_eva, col_agricola].fillna(0)

anual = anual.join(fijas).join(municipios).reset_index()
anual["tasa_def_ideam"] = anual.def_ideam_ha / anual.bosque_ideam_ha

en = lambda df, col, anio="anio": df.loc[df[anio].between(ANIO_INICIO, ANIO_FIN), col].sum()
print(f"municipio_anio: {anual.shape[0]:,} filas x {anual.shape[1]} columnas")
pd.DataFrame({"en la fuente": [en(ideam, "def_ha"), en(fin, "valor_mm"),
                               en(eva_anual, "Cosechada"), en(ica_anual, "total_bovinos")],
              "en la tabla": [anual.def_ideam_ha.sum(), anual.credito_mm.sum(),
                              anual.area_cosechada_ha.sum(), anual.bovinos.sum()]},
             index=["IDEAM def_ha", "FINAGRO millones", "EVA ha cosechadas", "ICA bovinos"]
             ).assign(porcentaje=lambda d: (100 * d["en la tabla"] / d["en la fuente"]).round(2))

## 7. Municipio × mes

Alertas de GFW y crédito de FINAGRO por municipio y mes, de `MES_INICIO` a `MES_FIN`, los meses en que las
dos fuentes están completas.

In [ ]:
MESES = pd.date_range(MES_INICIO, MES_FIN, freq="MS")
alertas = gfw.groupby(["cod_dane", "periodo"]).area_def_ha.sum().rename("alertas_gfw_ha")
alertas.index = alertas.index.set_names(["cod_dane", "mes"])

esqueleto = pd.MultiIndex.from_product([codigos, MESES], names=["cod_dane", "mes"])
mensual = pd.DataFrame(index=esqueleto).join(alertas).join(credito(fin[fin.mes.isin(MESES)], "mes"))
col_credito = [c for c in mensual if c.startswith(("credito_", "n_creditos"))]
mensual[col_credito] = mensual[col_credito].fillna(0)
mensual = mensual.join(municipios[["region"]]).reset_index()

print(f"municipio_mes: {mensual.shape[0]:,} filas x {mensual.shape[1]} columnas")
pd.DataFrame({"en la fuente": [gfw.loc[gfw.periodo.isin(MESES), "area_def_ha"].sum(),
                               fin.loc[fin.mes.isin(MESES), "valor_mm"].sum()],
              "en la tabla": [mensual.alertas_gfw_ha.sum(), mensual.credito_mm.sum()]},
             index=["GFW ha", "FINAGRO millones"]
             ).assign(porcentaje=lambda d: (100 * d["en la tabla"] / d["en la fuente"]).round(2))

## 8. Municipio × cultivo × año

Hectáreas cosechadas de cada cultivo de la EVA, desde `ANIO_INICIO_CULTIVOS`. Los semestres A y B y las
variantes de un mismo cultivo (por ejemplo, arroz de riego y de secano) se suman en el año.

In [ ]:
cultivos_eva = eva[["Cod_mun", "Year", "Grupo_cultivo", "Cultivo", "Cosechada"]].copy()
cultivos_eva["cod_dane"] = cultivos_eva.Cod_mun.astype(str).str.zfill(5)
cultivos_eva["anio"] = cultivos_eva.Year.dt.year
cultivos_eva = cultivos_eva[cultivos_eva.anio >= ANIO_INICIO_CULTIVOS]
cultivos = (cultivos_eva.groupby(["cod_dane", "anio", "Grupo_cultivo", "Cultivo"], observed=True)
            .Cosechada.sum().rename("cosechada_ha").reset_index()
            .rename(columns={"Grupo_cultivo": "grupo_cultivo", "Cultivo": "cultivo"}))
cultivos[["grupo_cultivo", "cultivo"]] = cultivos[["grupo_cultivo", "cultivo"]].astype(str)

print(f"municipio_cultivo_anio: {cultivos.shape[0]:,} filas")
pd.DataFrame({"en la fuente": [cultivos_eva.loc[cultivos_eva.anio.between(ANIO_INICIO, ULTIMO_ANIO_EVA), "Cosechada"].sum()],
              "en la tabla": [cultivos.loc[cultivos.anio.between(ANIO_INICIO, ULTIMO_ANIO_EVA), "cosechada_ha"].sum()],
              "en municipio_anio": [anual.loc[anual.anio.between(ANIO_INICIO, ULTIMO_ANIO_EVA), "area_cosechada_ha"].sum()]},
             index=[f"EVA ha cosechadas {ANIO_INICIO}-{ULTIMO_ANIO_EVA}"])

## 9. Guardar en BigQuery

In [ ]:
DESCRIPCION_ANUAL = {
    "cod_dane": "Código DANE del municipio (5 dígitos).",
    "anio": "Año. Para el IDEAM, el año final del periodo (2020-2021 es 2021).",
    "def_ideam_ha": "Deforestación oficial del IDEAM en el municipio, en hectáreas.",
    "bosque_ideam_ha": "Bosque natural al inicio del periodo, según el IDEAM. Denominador de tasa_def_ideam.",
    "sin_info_ideam_ha": "Hectáreas que el IDEAM no pudo observar (nubes).",
    "alertas_gfw_ha": "Hectáreas con alerta de Global Forest Watch en el año.",
    "credito_mm": "Crédito de FINAGRO desembolsado en el municipio, todos los eslabones, en millones de pesos.",
    "credito_produccion_mm": "Crédito de FINAGRO del eslabón de producción, en millones de pesos.",
    "credito_arroz_produccion_mm": "Crédito de producción a la cadena del arroz, en millones de pesos.",
    "credito_cacao_produccion_mm": "Crédito de producción a la cadena del cacao, en millones de pesos.",
    "credito_cafe_produccion_mm": "Crédito de producción a la cadena del café, en millones de pesos.",
    "credito_ganaderia_produccion_mm": "Crédito de producción a la ganadería (doble propósito, leche y carne), "
                                       "en millones de pesos.",
    "credito_palma_produccion_mm": "Crédito de producción a la palma de aceite, en millones de pesos.",
    "area_cosechada_ha": "Hectáreas cosechadas de todos los cultivos (EVA). Vacío después del último año de la EVA.",
    "cosechada_palma_ha": "Hectáreas cosechadas de palma de aceite (EVA).",
    "cosechada_cacao_ha": "Hectáreas cosechadas de cacao (EVA).",
    "cosechada_cafe_ha": "Hectáreas cosechadas de café (EVA).",
    "cosechada_arroz_ha": "Hectáreas cosechadas de arroz (EVA).",
    "bovinos": "Inventario de bovinos (ICA).",
    "fincas_bovinas": "Fincas con bovinos (ICA).",
    "bosque_ideam_inicial_ha": "Bosque natural del IDEAM al inicio del primer periodo. Fijo para cada municipio.",
    "n_celdas": "Celdas de 5 km del municipio en los paneles (las de al menos 50 ha de bosque base).",
    "area_grilla_ha": "n_celdas x 2.500 ha: área del municipio cubierta por esas celdas.",
    "nombre_departamento": "Departamento (tabla de municipios del Observatorio).",
    "nombre_municipio": "Nombre del municipio (tabla de municipios del Observatorio).",
    "region": "Región (tabla de municipios del Observatorio).",
    "ruralidad": "Categoría de ruralidad del DNP (tabla de municipios del Observatorio).",
    "tasa_def_ideam": "def_ideam_ha / bosque_ideam_ha: fracción del bosque natural deforestada en el año.",
}
guardar_tabla(
    anual, "municipio_anio",
    f"Panel municipal por año ({ANIO_INICIO}-{ANIO_FIN}): deforestación (IDEAM y GFW), crédito de FINAGRO, "
    "cultivos de la EVA, ganado del ICA y atributos del municipio. Es la tabla de la segmentación y los modelos.",
    descripciones=DESCRIPCION_ANUAL,
    procedencia={"fuente": "panel_ideam, panel_deforestacion_colombia y tablas del Observatorio",
                 "finagro": TABLA_FINAGRO, "eva": TABLA_EVA, "ica": TABLA_ICA, "municipios": TABLA_MUNICIPIOS,
                 "anios": f"{ANIO_INICIO}-{ANIO_FIN}", "ultimo_anio_eva": ULTIMO_ANIO_EVA})

guardar_tabla(
    mensual, "municipio_mes",
    f"Panel municipal por mes ({MESES[0]:%Y-%m} a {MESES[-1]:%Y-%m}): alertas de GFW y crédito de FINAGRO.",
    descripciones={**{k: v for k, v in DESCRIPCION_ANUAL.items() if k.startswith("credito")},
                   "cod_dane": DESCRIPCION_ANUAL["cod_dane"], "mes": "Primer día del mes.",
                   "alertas_gfw_ha": "Hectáreas con alerta de Global Forest Watch en el mes.",
                   "region": DESCRIPCION_ANUAL["region"]},
    procedencia={"fuente": "panel_deforestacion_colombia y FINAGRO", "finagro": TABLA_FINAGRO,
                 "meses": f"{MES_INICIO} a {MES_FIN}"})

guardar_tabla(
    cultivos, "municipio_cultivo_anio",
    f"Hectáreas cosechadas por municipio, cultivo y año según la EVA, desde {ANIO_INICIO_CULTIVOS}.",
    descripciones={"cod_dane": DESCRIPCION_ANUAL["cod_dane"], "anio": "Año.",
                   "grupo_cultivo": "Grupo de cultivo de la EVA.", "cultivo": "Cultivo de la EVA.",
                   "cosechada_ha": "Hectáreas cosechadas en el año (semestres sumados)."},
    procedencia={"fuente": "EVA", "eva": TABLA_EVA, "desde": ANIO_INICIO_CULTIVOS})

Listo: quedan cargadas todas las tablas. Siguen los notebooks de análisis, del 07 en adelante.